In [1]:
import geopandas as gpd
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import os
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.cm import ScalarMappable
import matplotlib.ticker as mticker
from shapely import wkb
import numpy as np

save_dir = "plots/analysis_plots"
os.makedirs(save_dir, exist_ok=True)

In [2]:
# Load warning areas shapefile
zone_shapefile_path = "../1_data/raw/WarningAreas/WarningAreas.shp"
gdf_zones = gpd.read_file(zone_shapefile_path)
gdf_zones = gdf_zones[gdf_zones.geometry.is_valid]
gdf_zones

,id,Sottozona,geometry
0,5,Cala5,"POLYGON ((629939.244 4411525.635, 629938.285 4..."
1,7,Cala7,"POLYGON ((636182.865 4274575.855, 636199.835 4..."
2,2,Cala2,"POLYGON ((622663.501 4332282.361, 622617.46 43..."
3,4,Cala4,"MULTIPOLYGON (((588209.1 4230865.079, 588205.1..."
4,3,Cala3,"POLYGON ((613282.231 4275683.466, 613252.157 4..."
5,6,Cala6,"POLYGON ((664382.815 4375997.552, 664390.149 4..."
6,8,Cala8,"POLYGON ((585805.641 4197281.855, 585774.897 4..."
7,1,Cala1,"MULTIPOLYGON (((567553.637 4412140.529, 567553..."


In [ ]:
# Extract grid attributes
climate_file_path = "../../Calabria_dataset/InputReteGood/Climatic/2017/20170701.h5"

with h5py.File(climate_file_path, "r") as h5_file:
    attributes_table = h5_file["attributes/table"][:]

attribute_names = [attr[0].decode() for attr in attributes_table]
attribute_values = [attr[1][0] for attr in attributes_table]

attributes_dict = dict(zip(attribute_names, attribute_values))

ncols = int(attributes_dict["ncols"])
nrows = int(attributes_dict["nrows"])
xllcorner = attributes_dict["xllcorner"]
yllcorner = attributes_dict["yllcorner"]
cellsize = attributes_dict["cellsize"]

attributes_dict

In [ ]:
ncols, nrows, xllcorner, yllcorner, cellsize

In [ ]:
# Compute real-world coordinates for each grid cell
cell_coordinates = []

for row in range(nrows):
    for col in range(ncols):
        x_coord = xllcorner + (col * cellsize)
        y_coord = yllcorner + (row * cellsize)

        cell_coordinates.append((row, col, x_coord, y_coord))

df_cell_coordinates = pd.DataFrame(cell_coordinates, columns=["Row", "Column", "X_Coord", "Y_Coord"])

In [ ]:
df_cell_coordinates

In [ ]:
# Convert to GeoDataFrame
df_cell_coordinates = gpd.GeoDataFrame(df_cell_coordinates, 
                                       geometry=gpd.points_from_xy(df_cell_coordinates.X_Coord, df_cell_coordinates.Y_Coord), 
                                       crs="EPSG:32633")

In [ ]:
df_cell_coordinates

In [ ]:
# Assign each grid cell to a warning zone
gdf_zones_bounds = gdf_zones.total_bounds 

df_cell_coordinates = df_cell_coordinates[
    (df_cell_coordinates["X_Coord"] >= gdf_zones_bounds[0]) &
    (df_cell_coordinates["X_Coord"] <= gdf_zones_bounds[2]) &
    (df_cell_coordinates["Y_Coord"] >= gdf_zones_bounds[1]) &
    (df_cell_coordinates["Y_Coord"] <= gdf_zones_bounds[3])
]

df_cell_zones = df_cell_coordinates.sjoin(gdf_zones, how="left", predicate="intersects")

In [ ]:
df_cell_zones

In [ ]:
# Clean and save the final zone mapping
df_cell_zones = df_cell_zones.rename(columns={"id": "Zone_ID"}).drop(columns=["Sottozona", "index_right"])
df_cell_zones = df_cell_zones.dropna(subset=["Zone_ID"]).reset_index(drop=True)
df_cell_zones.to_parquet("../1_data/processed/cell_zones.parquet")

In [ ]:
# Load precomputed cell-zone mapping
df_cell_zones = pd.read_parquet("../1_data/processed/cell_zones.parquet")
df_cell_zones

In [ ]:
# Plot cleaned grid cells
cmap = ListedColormap(plt.colormaps["Set1"].colors[:8])

fig, ax = plt.subplots(figsize=(9, 9))
gdf_fire_zones = gpd.GeoDataFrame(df_cell_zones, 
                                     geometry=gpd.points_from_xy(df_cell_zones.X_Coord, 
                                                                 df_cell_zones.Y_Coord), 
                                     crs="EPSG:32633")

gdf_fire_zones.plot(column="Zone_ID", cmap=cmap, legend=True, ax=ax, markersize=50, alpha=0.7)

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x * 1e-6:.2f}M"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y * 1e-6:.2f}M"))
plt.title("Cleaned Grid Cells")
plt.xlabel("Easting (millions of meters)")
plt.ylabel("Northing (millions of meters)")
plt.grid(True, linestyle="--", alpha=0.1)
plot_path = os.path.join(save_dir, "cleaned_grid_cells.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')


In [ ]:
zone_ids = sorted(df_cell_zones["Zone_ID"].unique())
n_zones = len(zone_ids)
cmap = ListedColormap(plt.colormaps["Set1"].colors[:n_zones])
bounds = np.arange(1, n_zones + 2) 
norm = BoundaryNorm(bounds, cmap.N)

gdf_fire_zones = gpd.GeoDataFrame(
    df_cell_zones, 
    geometry=gpd.points_from_xy(df_cell_zones.X_Coord, df_cell_zones.Y_Coord),
    crs="EPSG:32633"
)


fig, ax = plt.subplots(figsize=(6, 6), constrained_layout=True)

gdf_fire_zones.plot(
    column="Zone_ID",
    cmap=cmap,
    norm=norm,
    legend=False,
    ax=ax,
    markersize=1  
)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")


tick_positions = np.arange(1, n_zones + 1) + 0.5
tick_labels = [str(i) for i in zone_ids]

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax, boundaries=bounds, ticks=tick_positions)
cbar.ax.set_yticklabels(tick_labels)
cbar.set_label("Zone ID", labelpad=10)

plot_path = os.path.join(save_dir, "cleaned_grid_cells_final.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.close()
